In [5]:
import duckdb
import requests
import json

# Setup paths and configurations
DB_PATH = "../data/omop_clinical.duckdb"
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "qwen2.5-coder:7b"
SIMILARITY_THRESHOLD = 0.90

def get_unmapped_drugs():
    """Fetches unique unmapped drug descriptions from the database."""
    with duckdb.connect(DB_PATH) as con:
        # We exclude 'Unknown' because if the source literally had no text, the AI cannot guess it
        return con.execute("""
            SELECT DISTINCT drug_source_value 
            FROM drug_exposure 
            WHERE drug_concept_id = 0 
              AND drug_source_value IS NOT NULL 
              AND drug_source_value != 'Unknown'
        """).fetchall()

def ask_llm(prompt):
    """Sends a deterministic prompt to the local LLM."""
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.0} # Enforce strict deterministic output
    }
    try:
        response = requests.post(OLLAMA_URL, json=payload)
        response.raise_for_status()
        return response.json().get("response", "").strip()
    except requests.exceptions.RequestException as e:
        print(f"LLM API Error: {e}")
        return None

def normalize_drug(raw_text):
    """Uses LLM to extract the core active ingredient from messy text."""
    prompt = f"""
    You are an expert clinical data encoder. 
    Extract the core medication or active ingredient name from the following raw text. 
    Remove any dosages, grammatical noise, or administrative tags.
    Return ONLY the clean medication name, nothing else.
    
    Raw Text: {raw_text}
    Clean Medication Name:"""
    return ask_llm(prompt)

def find_best_rxnorm_match(con, clean_text):
    """Uses DuckDB's Jaro-Winkler similarity to find the best RxNorm match."""
    query = """
        SELECT concept_id, concept_name, jaro_winkler_similarity(LOWER(concept_name), LOWER(?)) as score
        FROM concept
        WHERE vocabulary_id = 'RxNorm' 
          AND domain_id = 'Drug'
          AND standard_concept = 'S' -- Only map to Standard concepts
        ORDER BY score DESC
        LIMIT 1
    """
    result = con.execute(query, (clean_text,)).fetchone()
    
    if result and result[2] >= SIMILARITY_THRESHOLD:
        return result[0], result[1], result[2]
    return None, None, None

# EXECUTION BLOCK
print("🤖 STARTING AI SEMANTIC MAPPING (DRUGS) [GOD MODE]\n" + "-"*50)

unmapped = get_unmapped_drugs()
print(f"🔍 Found {len(unmapped)} unique unmapped drug descriptions.")

if not unmapped:
    print("✅ No unmapped valid drugs found. The database is already fully standardized!")
else:
    successful_mappings = []

    with duckdb.connect(DB_PATH) as con:
        for row in unmapped:
            raw_text = row[0]
            print(f"\n⚙️ Processing: '{raw_text}'")
            
            # 1. Ask LLM to normalize
            clean_text = normalize_drug(raw_text)
            if not clean_text:
                continue
            print(f"   🧠 LLM Normalized: '{clean_text}'")
            
            # 2. Check DuckDB for Jaro-Winkler match in RxNorm
            concept_id, concept_name, score = find_best_rxnorm_match(con, clean_text)
            
            if concept_id:
                print(f"   ✅ Match Found: {concept_name} (ID: {concept_id}) | Confidence: {score:.2f}")
                successful_mappings.append((concept_id, raw_text))
            else:
                print(f"   ❌ No match met the {SIMILARITY_THRESHOLD} threshold.")
        
        # 3. Bulk Update the Database
        if successful_mappings:
            print(f"\n💾 Writing {len(successful_mappings)} mapped concepts back to the database...")
            con.executemany("""
                UPDATE drug_exposure
                SET drug_concept_id = ?
                WHERE drug_source_value = ? 
                  AND drug_concept_id = 0
            """, successful_mappings)
            print("✅ Database successfully updated!")
        else:
            print("\n⚠️ No new mappings met the confidence criteria to write back.")

🤖 STARTING AI SEMANTIC MAPPING (DRUGS) [GOD MODE]
--------------------------------------------------
🔍 Found 0 unique unmapped drug descriptions.
✅ No unmapped valid drugs found. The database is already fully standardized!
